# optimizer-class-dispatch — worked example 2: Dispatch with per-optimizer extra keyword arguments

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-class-dispatch`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Different optimizers accept different hyperparameters beyond `lr` — Adam needs `betas`, RMSprop needs `alpha`, SGD can take `momentum`. A factory that accepts `**kwargs` and passes them to the constructor lets the caller specify any optimizer-specific arguments via a config dict without the factory needing to know what they are.

## Worked solution

**Step 1 — Pass `**kwargs` through.**
The factory signature is `build_optimizer(name, params, lr, **kwargs)`. We look up the class, then call `cls(params, lr=lr, **kwargs)`. The extra keyword arguments flow directly to the optimizer's `__init__`.

**Step 2 — Adam with custom betas.**
We call `build_optimizer('adam', params, lr=1e-3, betas=(0.5, 0.9))`. The factory passes `betas=(0.5, 0.9)` to `torch.optim.Adam`. We verify the optimizer stored our custom betas (accessible via `opt.defaults['betas']`).

**Step 3 — SGD with momentum.**
`build_optimizer('sgd', params, lr=0.1, momentum=0.9)` passes `momentum=0.9` to `torch.optim.SGD`. We verify `opt.defaults['momentum'] == 0.9`.

**Step 4 — Plain call with no extras.**
Calling with no extra kwargs still works because `**kwargs` is empty, and the optimizers all have defaults for their additional hyperparameters.

In [ ]:
import torch as t
import torch.nn as nn

OPTIM_MAP = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def build_optimizer(name: str, params, lr: float, **kwargs) -> t.optim.Optimizer:
    cls = OPTIM_MAP[name]
    return cls(params, lr=lr, **kwargs)

# --- exercise it ---
t.manual_seed(0)
model = nn.Linear(16, 8)

# Plain (no extras)
opt_plain = build_optimizer('adam', model.parameters(), lr=1e-3)
print(f'Adam default betas: {opt_plain.defaults["betas"]}')

# With custom betas
opt_betas = build_optimizer('adam', model.parameters(), lr=1e-3, betas=(0.5, 0.9))
print(f'Adam custom  betas: {opt_betas.defaults["betas"]}')
assert opt_betas.defaults['betas'] == (0.5, 0.9)

# SGD with momentum
opt_sgd = build_optimizer('sgd', model.parameters(), lr=0.01, momentum=0.9)
print(f'SGD momentum: {opt_sgd.defaults["momentum"]}')
assert opt_sgd.defaults['momentum'] == 0.9

print('All checks passed.')